# C2 — Plant Disease Model Training (Clean Submission Notebook)

**Models:** FasterNet, EfficientViT and GhostNet. **Training/validation:** PlantVillage (16,745/4,182 images, 9 classes). **Reserved C3 test set:** 777 PlantDoc field images — **do not use for C2 training, validation, tuning or model selection**.

This is a reproducible *clean copy* of the original C2 notebook. The original notebook and its historical execution outputs should be retained separately. This notebook **does not train models automatically**: training is an optional manual step, and saved checkpoint analysis is independent of training.

**Dependencies:** teammate-provided `c1_pipeline.py`, `models.py`, the dataset folder (`splits/`, `config/`, `prepared_data/`) and standard Python ML packages. Check your team's `requirements.txt` before installation.

## 1. Connect Drive and check the execution environment

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import torch

DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_ROOT = DRIVE_ROOT / 'Plant_Disease_HW2_C2'
IMAGE_ROOT = PROJECT_ROOT / 'prepared_data'
CHECKPOINT_DIR = DRIVE_ROOT / 'C2_Model_Checkpoints'
GRAPH_DIR = DRIVE_ROOT / 'C2_Training_Graphs'
FINAL_DIR = DRIVE_ROOT / 'C2_Final_Deliverables'

assert PROJECT_ROOT.is_dir(), f'Dataset folder not found: {PROJECT_ROOT}'
for rel in ['splits/train.csv', 'splits/valid.csv', 'config/class_to_idx.json', 'prepared_data/train', 'prepared_data/valid']:
    assert (PROJECT_ROOT / rel).exists(), f'Missing: {rel}'
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('Dataset folder:', PROJECT_ROOT)

## 2. Restore the team repository

This notebook requires **the team's original helper scripts**, which are not embedded in the notebook. Easiest: place the extracted team repository in `/content/Plant-Disease-Classification-HW2-6.-main`, or upload the repository ZIP in Colab's file sidebar and run the extraction below. Alternatively, update `REPO_PATH` to the location of a mounted Drive copy. Do **not** upload your dataset or checkpoint weights to a public GitHub repository.

In [ ]:
import sys, zipfile

REPO_PATH = Path('/content/Plant-Disease-Classification-HW2-6.-main')
ZIP_PATH = Path('/content/Plant-Disease-Classification-HW2-6.-main.zip')

if not REPO_PATH.is_dir() and ZIP_PATH.is_file():
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall('/content/')

if not (REPO_PATH / 'c1_pipeline.py').exists() or not (REPO_PATH / 'models.py').exists():
    raise FileNotFoundError('Upload/extract the team repository ZIP or set REPO_PATH to the directory containing c1_pipeline.py and models.py.')

if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))

from c1_pipeline import create_dataloaders
from models import initialize_model, MODEL_NAMES
print('Model architectures:', MODEL_NAMES)

## 3. Copy PlantVillage training/validation images to fast Colab storage

In [ ]:
import shutil, time
LOCAL_IMAGE_ROOT = Path('/content/plant_data')

start = time.time()
for split in ['train', 'valid']:
    source = IMAGE_ROOT / split
    destination = LOCAL_IMAGE_ROOT / split
    if not destination.is_dir():
        print('Copying', split, flush=True)
        shutil.copytree(source, destination)
    else:
        print(split, 'already exists locally; verify its contents if restoring an interrupted copy.')
print(f'Local copy check finished in {(time.time() - start)/60:.2f} min')

# The C1 helper also constructs a test DataLoader; C2 must never iterate it.
data = create_dataloaders(project_root=PROJECT_ROOT, image_root=LOCAL_IMAGE_ROOT,
                          batch_size=32, num_workers=0)
train_loader, valid_loader = data['train'], data['valid']
assert len(train_loader.dataset) == 16745, 'Unexpected training split size'
assert len(valid_loader.dataset) == 4182, 'Unexpected validation split size'
print('Training:', len(train_loader.dataset), 'Validation:', len(valid_loader.dataset))

## 4. Hyperparameters and one-epoch functions

In [ ]:

import torch
import random
import numpy as np

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Training configuration
BATCH_SIZE = 32
MAX_EPOCHS = 30
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 7

# Device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Training configuration ready!")
print("Device:", device)
print("Batch size:", BATCH_SIZE)
print("Maximum epochs:", MAX_EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Early stopping patience:", EARLY_STOPPING_PATIENCE)


import torch

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size

        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += batch_size

    return total_loss / total, correct / total


@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size

        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += batch_size

    return total_loss / total, correct / total


print("Training and validation functions are ready!")


## 5. Checkpoint saving and recovery

In [ ]:

from pathlib import Path
import torch

# Save checkpoints in your own Google Drive
CHECKPOINT_DIR = Path(
    "/content/drive/MyDrive/C2_Model_Checkpoints"
)

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def save_checkpoint(
    model,
    optimizer,
    scheduler,
    epoch,
    val_loss,
    val_accuracy,
    best_val_loss,
    patience_counter,
    history,
    model_name,
    checkpoint_type
):
    checkpoint = {
        "model_name": model_name,
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
        "best_val_loss": best_val_loss,
        "patience_counter": patience_counter,
        "history": history,
    }

    filename = (
        f"{model_name.lower()}_{checkpoint_type}.pth"
    )

    save_path = CHECKPOINT_DIR / filename

    # Save to a temporary file first
    temporary_path = save_path.with_suffix(".tmp")
    torch.save(checkpoint, temporary_path)
    temporary_path.replace(save_path)

    return save_path

print("Checkpoint directory:", CHECKPOINT_DIR)
print("Automatic checkpoint saving is ready!")


import torch

def load_checkpoint(model, optimizer, scheduler, model_name):
    checkpoint_path = (
        CHECKPOINT_DIR / f"{model_name.lower()}_latest.pth"
    )

    if not checkpoint_path.exists():
        print(f"No previous checkpoint for {model_name}.")
        print("Training will start from epoch 1.")
        return 1, float("inf"), 0, {
            "train_loss": [],
            "train_accuracy": [],
            "val_loss": [],
            "val_accuracy": []
        }

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=False
    )

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    start_epoch = checkpoint["epoch"] + 1
    best_val_loss = checkpoint["best_val_loss"]
    patience_counter = checkpoint["patience_counter"]
    history = checkpoint["history"]

    print(f"Resuming {model_name} from epoch {start_epoch}")

    return start_epoch, best_val_loss, patience_counter, history

print("Checkpoint recovery function is ready!")


## 6. Final training loop (one definition; safely resumes from latest checkpoint)

In [ ]:

import time
import torch
import torch.nn as nn
import torch.optim as optim

def run_training(model_name, max_epochs=MAX_EPOCHS):
    print(f"\nStarting training: {model_name}")

    model = initialize_model(
        architecture=model_name,
        num_classes=9,
        pretrained=True
    ).to(device)

    optimizer = optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    criterion = nn.CrossEntropyLoss()

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=3
    )

    # Automatically resume if a checkpoint exists
    start_epoch, best_val_loss, patience_counter, history = (
        load_checkpoint(model, optimizer, scheduler, model_name)
    )

    if patience_counter >= EARLY_STOPPING_PATIENCE:
        print('Early stopping was already reached; no more epochs will run.')
        return history

    if start_epoch > max_epochs:
        print("Training already completed the requested epochs.")
        return history

    for epoch in range(start_epoch, max_epochs + 1):
        start_time = time.time()

        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, criterion, device
        )

        val_loss, val_acc = validate_one_epoch(
            model, valid_loader, criterion, device
        )

        scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_acc)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0

            save_checkpoint(
                model, optimizer, scheduler,
                epoch, val_loss, val_acc,
                best_val_loss, patience_counter,
                history, model_name, "best"
            )

            print("Best model checkpoint saved!")
        else:
            patience_counter += 1

        save_checkpoint(
            model, optimizer, scheduler,
            epoch, val_loss, val_acc,
            best_val_loss, patience_counter,
            history, model_name, "latest"
        )

        elapsed = (time.time() - start_time) / 60

        print(
            f"Epoch {epoch}/{max_epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.2%} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.2%} | "
            f"Time: {elapsed:.1f} min",
            flush=True
        )

        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print("Early stopping triggered.")
            break

    return history

print("Automatic training recovery is enabled!")


## 7. Optional: train or resume a model **manually**

**Do not run this unless additional training is actually required.** All three models were already trained in the original experiment. Their checkpoints, not a new training run, should be used to generate the C2 report. If resuming a checkpoint, the code restores model, optimizer, scheduler and history from `_latest.pth`.

In [ ]:
# Uncomment only if you intentionally need to train/resume:
# fasternet_history = run_training('FasterNet', max_epochs=30)
# efficientvit_history = run_training('EfficientViT', max_epochs=30)
# ghostnet_history = run_training('GhostNet', max_epochs=30)
print('Training is disabled by default. Existing checkpoints are used below.')

## 8. Load saved histories and compare best checkpoints

In [ ]:
import pandas as pd
histories, results = {}, []
for name in ['FasterNet','EfficientViT','GhostNet']:
    best_path = CHECKPOINT_DIR / f'{name.lower()}_best.pth'
    latest_path = CHECKPOINT_DIR / f'{name.lower()}_latest.pth'
    if not (best_path.is_file() and latest_path.is_file()):
        raise FileNotFoundError(f'Missing checkpoint for {name}: {best_path} / {latest_path}')
    # Load only checkpoint files produced by your own trusted training code.
    best = torch.load(best_path, map_location='cpu', weights_only=False)
    latest = torch.load(latest_path, map_location='cpu', weights_only=False)
    histories[name] = latest['history']
    results.append({'Model':name, 'Epochs Trained':latest['epoch'],
                    'Best Epoch':best['epoch'],
                    'Best Validation Accuracy (%)':round(best['val_accuracy']*100,2),
                    'Best Validation Loss':round(best['val_loss'],4)})
comparison_df = pd.DataFrame(results)
display(comparison_df)
comparison_df.to_csv(DRIVE_ROOT / 'C2_Model_Comparison.csv', index=False)

## 9. Recreate the six training curves

In [ ]:
import matplotlib.pyplot as plt
GRAPH_DIR.mkdir(parents=True, exist_ok=True)
for model_name, history in histories.items():
    epochs = list(range(1, len(history['train_loss']) + 1))
    for metric, ylabel, transform in [('accuracy','Accuracy (%)',100),('loss','Loss',1)]:
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.plot(epochs, [x*transform for x in history['train_'+metric]], marker='o', label='Training')
        ax.plot(epochs, [x*transform for x in history['val_'+metric]], marker='o', label='Validation')
        ax.set(title=f'{model_name} - {metric.title()}', xlabel='Epoch', ylabel=ylabel)
        ax.grid(alpha=0.3)
        ax.legend()
        fig.tight_layout()
        fig.savefig(GRAPH_DIR / f'{model_name}_{metric}.png', dpi=300)
        plt.show()
        plt.close(fig)
print('Six graphs saved to:', GRAPH_DIR)

## 10. Validation integrity check (recommended before reporting)

The very high PlantVillage validation accuracy merits checking for overlap. The checks below compare the **CSV identifiers**; they do *not* rule out visually similar or near-duplicate images. A stronger perceptual/hash audit may require collaboration with C1 and must not use the C3 test set.

In [ ]:
import pandas as pd
train_df = pd.read_csv(PROJECT_ROOT / 'splits/train.csv')
valid_df = pd.read_csv(PROJECT_ROOT / 'splits/valid.csv')
print('Train rows:', len(train_df), '| Valid rows:', len(valid_df))
print('Train columns:', train_df.columns.tolist())
if {'class_name','file_id'}.issubset(train_df.columns) and {'class_name','file_id'}.issubset(valid_df.columns):
    left = set(zip(train_df['class_name'].astype(str),train_df['file_id'].astype(str)))
    right = set(zip(valid_df['class_name'].astype(str),valid_df['file_id'].astype(str)))
    print('Shared (class_name, file_id) identifiers:', len(left & right))
else:
    print('Cannot compare identifiers: expected columns missing.')
print('This check does not establish absence of image-content or near-duplicate leakage.')

## Submission notes

- **Best checkpoint selection:** minimum validation loss (not necessarily maximum accuracy).
- **Results:** FasterNet best epoch 12 (99.90%, 0.0049), EfficientViT best epoch 16 (99.95%, 0.0035), GhostNet best epoch 11 (99.90%, 0.0048). These are *PlantVillage validation* results only.
- **No C3 test use:** The `plantdoc_test` data are reserved for subsequent field-image evaluation; do not iterate the test DataLoader while preparing C2.
- **Team files required:** `c1_pipeline.py`, `models.py`, `requirements.txt`, dataset metadata/config and accessible prepared images. The helper scripts and dataset are **not embedded** in this notebook.
- **Checkpoint location:** `/content/drive/MyDrive/C2_Model_Checkpoints`. Existing saved checkpoints are retained, and this notebook does not overwrite them unless you manually trigger training.
- **Versioning:** Include the team's code version and package versions in your final team handoff. Don't upload large `.pth` checkpoints to GitHub unless your team specifically uses Git LFS; share Drive paths instead.
- **Original record:** Keep the original executed notebook unchanged as evidence of actual training runs; this clean notebook is a reproducible presentation copy.